In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType

def mask_email(email):
    if email is None or '@' not in email:
        return email
    parts = email.split('@')
    username = parts[0]
    domain = parts[1]
    
    if len(username) <= 2:
        masked_username = '*' * len(username)
    else:
        masked_username = username[0] + '*' * (len(username) - 2) + username[-1]
    
    return f"{masked_username}@{domain}"

mask_email_udf = udf(mask_email, StringType())

df_customers = spark.table("customer.transaction.gold_dim_customer")
df_email_masked = df_customers.select(
    "customer_id",
    col("email").alias("original_email"),
    mask_email_udf(col("email")).alias("masked_email")
)

print("Email masking function created")
display(df_email_masked.limit(10))

In [0]:
def mask_phone(phone):
    if phone is None:
        return phone
    digits = ''.join(filter(str.isdigit, phone))
    if len(digits) >= 4:
        return f"(***) ***-{digits[-4:]}"
    return "***-****"

mask_phone_udf = udf(mask_phone, StringType())

df_phone_masked = df_customers.select(
    "customer_id",
    col("phone").alias("original_phone"),
    mask_phone_udf(col("phone")).alias("masked_phone")
)

print("Phone masking function created")
display(df_phone_masked.limit(10))

In [0]:
def apply_role_based_masking(df, role):
    if role == "ADMIN":
        return df.select("customer_id", "full_name", "email", "phone", "customer_segment")
    
    elif role == "ANALYST":
        return df.select(
            "customer_id",
            "full_name",
            mask_email_udf(col("email")).alias("email"),
            mask_phone_udf(col("phone")).alias("phone"),
            "customer_segment"
        )
    
    else:
        return df.select(
            lit("***").alias("customer_id"),
            lit("***").alias("full_name"),
            lit("***@***.com").alias("email"),
            lit("(***) ***-****").alias("phone"),
            "customer_segment"
        )

current_role = "ANALYST"
df_masked = apply_role_based_masking(df_customers, current_role)

print(f"Applied masking for role: {current_role}")
display(df_masked.limit(10))

In [0]:
df_analyst_view = df_customers.select(
    "customer_id",
    "full_name",
    mask_email_udf(col("email")).alias("email"),
    mask_phone_udf(col("phone")).alias("phone"),
    "customer_segment"
)

df_analyst_view.createOrReplaceTempView("customer_masked_analyst")

df_analyst_view.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("customer.transaction.gold_dim_customer_masked")

print("Created customer.transaction.gold_dim_customer_masked")

In [0]:
import base64

def simple_encrypt(value):
    if value is None:
        return None
    return base64.b64encode(value.encode('utf-8')).decode('utf-8')

def simple_decrypt(encrypted_value):
    if encrypted_value is None:
        return None
    return base64.b64decode(encrypted_value.encode('utf-8')).decode('utf-8')

encrypt_udf = udf(simple_encrypt, StringType())
decrypt_udf = udf(simple_decrypt, StringType())

df_encrypted = df_customers.select(
    "customer_id",
    col("email").alias("email_plaintext"),
    encrypt_udf(col("email")).alias("email_encrypted"),
    col("phone").alias("phone_plaintext"),
    encrypt_udf(col("phone")).alias("phone_encrypted")
)

print("Applied column-level encryption (base64 demo)")
display(df_encrypted.limit(5))

df_decrypted = df_encrypted.select(
    "customer_id",
    "email_encrypted",
    decrypt_udf(col("email_encrypted")).alias("email_decrypted")
)

print("Decryption example:")
display(df_decrypted.limit(5))